In [10]:
import os
from pathlib import Path

# Get the directory of the notebook, then its parent (the project root)
# os.getcwd() is reliable in notebooks, but Path(__file__).resolve().parent 
# is preferred for scripts. We'll use a robust approach for the notebook.
PROJECT_ROOT = Path(os.getcwd()).parent

# Define directories based on the project structure
data_dir = PROJECT_ROOT / "data"
college_stats_dir = data_dir / "college_stats"
pff_data_dir = college_stats_dir / "pff_csvs"
cfbd_data_dir = college_stats_dir / "cfbd"
rosters_dir = college_stats_dir / "rosters"
passing_stats_dir = college_stats_dir / "passing"
rushing_stats_dir = college_stats_dir / "rushing"
receiving_stats_dir = college_stats_dir / "receiving"
kicking_stats_dir = college_stats_dir / "kicking"
punting_stats_dir = college_stats_dir / "punting"
defensive_stats_dir = college_stats_dir / "defense"
awards_stats_dir = college_stats_dir / "awards"
modeling_data_dir = data_dir / "modeling_datasets" / "college" / "aggregated_stats"
matching_data_dir = data_dir / "modeling_datasets" / "college" / "matching"

# Verify all dirs exist
for directory in [
    data_dir,
    college_stats_dir,
    pff_data_dir,
    cfbd_data_dir,
    rosters_dir,
    passing_stats_dir,
    rushing_stats_dir,
    receiving_stats_dir,
    kicking_stats_dir,
    punting_stats_dir,
    defensive_stats_dir,
    awards_stats_dir,
    modeling_data_dir,
    matching_data_dir,
]:
    directory.mkdir(parents=True, exist_ok=True)

# College Position Aggregation Scaffold

This notebook stage prepares college data for downstream custom score computation:
- Uses **games only** (no starts)
- Builds unique school lists for **PFF** and **Sports-Reference**
- Creates a **PFF → Sports-Reference team map skeleton**
- Outputs per-position, model-ready aggregated tables to `modeling_data_dir`

In [11]:
import re

import pandas as pd
from thefuzz import fuzz

YEAR_MIN = 2015
YEAR_MAX = 2025
FUZZ_THRESHOLD = 90

POS_MAP = {
    "PRO": "QB", "DUAL": "QB", "QB": "QB",
    "RB": "RB", "APB": "RB", "FB": "RB",
    "WR": "WR",
    "TE": "TE",
    "OT": "OL", "OG": "OL", "OC": "OL", "IOL": "OL", "OL": "OL", "C": "OL", "G": "OL", "T": "OL",
    "DT": "IDL", "DL": "IDL", "DI": "IDL", "NT": "IDL",
    "SDE": "EDGE", "WDE": "EDGE", "EDGE": "EDGE", "DE": "EDGE", "ED": "EDGE",
    "ILB": "LB", "OLB": "LB", "LB": "LB", "MLB": "LB",
    "CB": "DB", "S": "DB", "FS": "DB", "SS": "DB", "DB": "DB",
    "ATH": "ATH",
    "K": "SPEC", "P": "SPEC", "LS": "SPEC", "RET": "SPEC", "PK": "SPEC",
}
FINAL_POSITION_GROUPS = sorted({position for position in POS_MAP.values() if position and position != "ATH"})


def normalize_text(value: str) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_name(value: str) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = text.replace(".", "")
    text = re.sub(r"\b(jr|sr|ii|iii|iv|v|vi)\b", "", text)
    return normalize_text(text)


def normalize_school(value: str) -> str:
    return normalize_text(value)


def normalize_position(value: str) -> str:
    pos = normalize_text(value).upper()
    return POS_MAP.get(pos, pos)


def first_non_null(series: pd.Series):
    valid = series.dropna().astype(str).str.strip()
    valid = valid[valid != ""]
    return valid.iloc[0] if len(valid) else ""

In [6]:
# -----------------------------
# 1) Build Sports-Ref roster master + school list
# -----------------------------
roster_files = sorted(rosters_dir.glob("*_roster.csv"))
if not roster_files:
    raise FileNotFoundError(f"No roster files found in: {rosters_dir}")

filename_pattern = re.compile(r"^(.*)_(\d{4})_roster\.csv$")
roster_frames = []

for file_path in roster_files:
    roster_df = pd.read_csv(file_path, dtype=str).fillna("")

    match = filename_pattern.match(file_path.name)
    file_school = match.group(1).replace("_", " ").strip() if match else ""
    file_year = match.group(2) if match else ""

    if "School" not in roster_df.columns:
        roster_df["School"] = file_school
    else:
        roster_df["School"] = roster_df["School"].replace("", file_school)

    if "Year" not in roster_df.columns:
        roster_df["Year"] = file_year
    else:
        roster_df["Year"] = roster_df["Year"].replace("", file_year)

    roster_df["Year"] = pd.to_numeric(roster_df["Year"], errors="coerce")
    roster_df = roster_df[(roster_df["Year"] >= YEAR_MIN) & (roster_df["Year"] <= YEAR_MAX)].copy()

    roster_df["Player"] = roster_df.get("Player", "").astype(str).str.strip()
    roster_df["Player_ID"] = roster_df.get("Player_ID", "").astype(str).str.strip()
    roster_df["School"] = roster_df["School"].astype(str).str.strip()
    roster_df["pos"] = roster_df.get("pos", "").astype(str).str.strip()

    roster_df["name_norm"] = roster_df["Player"].map(normalize_name)
    roster_df["school_norm"] = roster_df["School"].map(normalize_school)
    roster_df["pos_norm"] = roster_df["pos"].map(normalize_position)
    roster_df["source_file"] = file_path.name

    roster_frames.append(roster_df)

roster_master = pd.concat(roster_frames, ignore_index=True)
roster_master = roster_master[roster_master["Player_ID"] != ""].copy()

sportsref_schools = (
    roster_master[["School", "school_norm"]]
    .drop_duplicates()
    .sort_values(["school_norm", "School"])
    .reset_index(drop=True)
)

# -----------------------------
# 2) Build PFF school list
# -----------------------------
pff_files = sorted(pff_data_dir.glob("pff_*_*.csv"))
if not pff_files:
    raise FileNotFoundError(f"No PFF files found in: {pff_data_dir}")

pff_frames = []
year_pattern = re.compile(r"_(\d{4})\.csv$")

for file_path in pff_files:
    year_match = year_pattern.search(file_path.name)
    file_year = int(year_match.group(1)) if year_match else None
    if file_year is None or file_year < YEAR_MIN or file_year > YEAR_MAX:
        continue

    pff_df = pd.read_csv(file_path, dtype=str).fillna("")
    pff_df["Year"] = file_year
    pff_df["source_file"] = file_path.name
    pff_df["player"] = pff_df.get("player", "").astype(str).str.strip()
    pff_df["team_name"] = pff_df.get("team_name", "").astype(str).str.strip()
    pff_df["position"] = pff_df.get("position", "").astype(str).str.strip()

    pff_df["name_norm"] = pff_df["player"].map(normalize_name)
    pff_df["team_norm"] = pff_df["team_name"].map(normalize_school)
    pff_df["pos_norm"] = pff_df["position"].map(normalize_position)

    pff_frames.append(pff_df)

pff_master = pd.concat(pff_frames, ignore_index=True)

pff_schools = (
    pff_master[["team_name", "team_norm"]]
    .drop_duplicates()
    .sort_values(["team_norm", "team_name"])
    .reset_index(drop=True)
)

'''# -----------------------------
# 3) Team map skeleton (PFF -> Sports-Ref)
# -----------------------------
sportsref_lookup = (
    sportsref_schools.groupby("school_norm")["School"]
    .agg(lambda values: sorted(set(values))[0])
    .to_dict()
)

team_map_skeleton = pff_schools.rename(columns={"team_name": "pff_school", "team_norm": "pff_school_norm"}).copy()
team_map_skeleton["sportsref_school"] = team_map_skeleton["pff_school_norm"].map(sportsref_lookup).fillna("")
team_map_skeleton["sportsref_school_norm"] = team_map_skeleton["sportsref_school"].map(normalize_school)
team_map_skeleton["match_method"] = team_map_skeleton["sportsref_school"].map(lambda value: "exact_norm" if value else "")
team_map_skeleton["map_status"] = team_map_skeleton["sportsref_school"].map(lambda value: "auto_mapped" if value else "needs_review")
team_map_skeleton["notes"] = ""

# Optional fuzzy suggestions for unresolved rows (non-destructive suggestions only)
sportsref_norm_values = sportsref_schools["school_norm"].dropna().tolist()
fuzzy_suggestions = []
for row in team_map_skeleton.itertuples(index=False):
    if row.sportsref_school:
        fuzzy_suggestions.append("")
        continue
    if not row.pff_school_norm:
        fuzzy_suggestions.append("")
        continue
    best_norm = ""
    best_score = 0
    for candidate in sportsref_norm_values:
        score = fuzz.ratio(row.pff_school_norm, candidate)
        if score > best_score:
            best_score = score
            best_norm = candidate
    if best_score >= FUZZ_THRESHOLD and best_norm in sportsref_lookup:
        fuzzy_suggestions.append(f"{sportsref_lookup[best_norm]} ({best_score})")
    else:
        fuzzy_suggestions.append("")

team_map_skeleton["fuzzy_suggestion"] = fuzzy_suggestions'''

# Save school inventories and team map scaffold
sportsref_school_path = matching_data_dir / "sportsref_unique_schools.csv"
pff_school_path = matching_data_dir / "pff_unique_schools.csv"
team_map_path = matching_data_dir / "pff_to_sportsref_team_map_skeleton.csv"
master_roster_path = matching_data_dir / "roster_master.csv"

sportsref_schools.to_csv(sportsref_school_path, index=False, encoding="utf-8-sig")
pff_schools.to_csv(pff_school_path, index=False, encoding="utf-8-sig")
roster_master.to_csv(master_roster_path, index=False, encoding="utf-8-sig")
#team_map_skeleton.to_csv(team_map_path, index=False, encoding="utf-8-sig")

print(f"Roster rows loaded: {len(roster_master):,}")
print(f"PFF rows loaded: {len(pff_master):,}")
print(f"Sports-Ref unique schools: {len(sportsref_schools):,}")
print(f"PFF unique schools: {len(pff_schools):,}")
#print(f"Team map rows needing review: {(team_map_skeleton['map_status'] == 'needs_review').sum():,}")
print(f"Saved: {sportsref_school_path}")
print(f"Saved: {pff_school_path}")
print(f"Saved: {team_map_path}")

Roster rows loaded: 161,584
PFF rows loaded: 110,912
Sports-Ref unique schools: 137
PFF unique schools: 138
Saved: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\college\aggregated_stats\sportsref_unique_schools.csv
Saved: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\college\aggregated_stats\pff_unique_schools.csv
Saved: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\college\aggregated_stats\pff_to_sportsref_team_map_skeleton.csv


In [ ]:
team_map_skeleton_path = matching_data_dir / "pff_to_sportsref_team_map_skeleton.csv"
if not team_map_skeleton_path.exists():
    raise FileNotFoundError(f"Team map skeleton not found. Please run the team mapping section first and save the skeleton to: {team_map_skeleton_path}")

# Check that each pff team in the skeleton maps to at most one sportsref team (to prevent destructive mappings)
mapping_df = pd.read_csv(team_map_skeleton_path, dtype=str).fillna("")
mapping_df["pff_school_norm"] = mapping_df["pff_school"].map(normalize_school)
mapping_df["sportsref_school_norm"] = mapping_df["sportsref_school"].map(normalize_school)
mapping_counts = mapping_df.groupby("pff_school_norm")["sportsref_school_norm"].nunique()
duplicates = mapping_counts[mapping_counts > 1]
if not duplicates.empty:
    duplicate_teams = mapping_counts[mapping_counts > 1].index.tolist()
    raise ValueError(f"Destructive mappings detected! The following PFF teams map to multiple Sports-Ref teams: {duplicate_teams}. Please resolve these before proceeding.")

print("Team map skeleton loaded successfully with no destructive mappings. Ready for review and finalization.")
print(mapping_counts)

Team map skeleton loaded successfully with no destructive mappings. Ready for review and finalization.
pff_school_norm
air force     1
akron         1
alabama       1
app state     1
arizona       1
             ..
wake          1
wash state    1
washington    1
wisconsin     1
wyoming       1
Name: sportsref_school_norm, Length: 138, dtype: int64


In [6]:
# Get unique positions in pff data
unique_pff_positions = pff_master["position"].dropna().unique()
print("Unique positions in PFF data:") 
print(unique_pff_positions)

Unique positions in PFF data:
['ED' 'CB' 'DI' 'S' 'LB' 'WR' 'HB' 'G' 'FB' 'TE' 'C' 'QB' '' 'ST' 'T' 'LS'
 'K' 'P']


In [ ]:
# -----------------------------
# 4) PFF matcher skeleton (year + school + side-of-ball + name)
# -----------------------------

def build_pff_matcher_skeleton(
    roster_df: pd.DataFrame,
    pff_df: pd.DataFrame,
    team_map_df: pd.DataFrame,
    school_fuzzy_threshold: int = 92,
    name_fuzzy_threshold: int = 88,
) -> pd.DataFrame:
    """
    Matcher scaffold with active fuzzy fallback:
      1) Applies team map (PFF -> Sports-Ref) using pff_school_norm -> sportsref_school_norm
      2) Exact match on (Year, school_norm, side_norm, name_norm)
      3) Fuzzy name fallback within (Year, school_norm, side_norm)

    Position-level requirement is intentionally relaxed to side-of-ball buckets.
    'ED' 'CB' 'DI' 'S' 'LB' 'WR' 'HB' 'G' 'FB' 'TE' 'C' 'QB' '' 'ST' 'T' 'LS'
    'K' 'P'
    """

    OFFENSE_POS = {"QB", "RB", "WR", "TE", "OL", "SPEC","OL","C","G","T","LS","K","P","HB","FB",""}
    DEFENSE_POS = {"IDL", "EDGE", "LB", "DB","ED","CB","DI","S","LB",""}

    def map_side(pos_norm: str) -> str:
        if pos_norm in OFFENSE_POS:
            return "OFF"
        if pos_norm in DEFENSE_POS:
            return "DEF"
        return ""

    map_df = team_map_df.copy()
    map_df["pff_school"] = map_df.get("pff_school", "").astype(str)
    map_df["pff_school_norm"] = map_df.get("pff_school_norm", "").astype(str).map(normalize_school)
    map_df["sportsref_school_norm"] = map_df.get("sportsref_school_norm", "").astype(str).map(normalize_school)

    roster_work = roster_df.copy()
    roster_work["school_norm"] = roster_work.get("School", "").map(normalize_school)
    roster_work["pos_norm"] = roster_work.get("pos", "").map(normalize_position)
    roster_work["side_norm"] = roster_work["pos_norm"].map(map_side)
    roster_work["name_norm"] = roster_work.get("Player", "").map(normalize_name)

    pff_work = pff_df.copy()
    pff_work["team_name"] = pff_work.get("team_name", "").astype(str)
    pff_work["team_norm"] = pff_work["team_name"].map(normalize_school)
    pff_work["pos_norm"] = pff_work.get("position", "").map(normalize_position)
    pff_work["side_norm"] = pff_work["pos_norm"].map(map_side)
    pff_work["name_norm"] = pff_work.get("player", "").map(normalize_name)

    roster_school_norm_to_name = (
        roster_work[["school_norm", "School"]]
        .drop_duplicates()
        .groupby("school_norm")["School"]
        .agg(lambda values: sorted(set(values))[0])
        .to_dict()
    )
    roster_school_norm_values = list(roster_school_norm_to_name.keys())

    norm_map = (
        map_df[map_df["sportsref_school_norm"].str.strip() != ""]
        .drop_duplicates(subset=["pff_school_norm"], keep="first")
        .set_index("pff_school_norm")["sportsref_school_norm"]
        .to_dict()
    )

    raw_map_to_name = (
        map_df[map_df.get("sportsref_school", "").astype(str).str.strip() != ""]
        .drop_duplicates(subset=["pff_school"], keep="first")
        .set_index("pff_school")["sportsref_school"]
        .to_dict()
        if "sportsref_school" in map_df.columns
        else {}
    )

    pff_work["mapped_school_norm"] = pff_work["team_norm"].map(norm_map).fillna("")

    unresolved_school_idx = pff_work.index[pff_work["mapped_school_norm"].astype(str).str.strip() == ""]
    for idx in unresolved_school_idx:
        pff_school_norm = str(pff_work.at[idx, "team_norm"]).strip()
        if not pff_school_norm:
            continue

        best_norm = ""
        best_score = -1
        for candidate in roster_school_norm_values:
            score = fuzz.ratio(pff_school_norm, candidate)
            if score > best_score:
                best_score = score
                best_norm = candidate

        if best_norm and best_score >= school_fuzzy_threshold:
            pff_work.at[idx, "mapped_school_norm"] = best_norm

    pff_work["mapped_school"] = pff_work["mapped_school_norm"].map(roster_school_norm_to_name).fillna("")
    unresolved_name_mask = pff_work["mapped_school"].astype(str).str.strip() == ""
    pff_work.loc[unresolved_name_mask, "mapped_school"] = pff_work.loc[unresolved_name_mask, "team_name"].map(raw_map_to_name).fillna("")
    unresolved_name_mask = pff_work["mapped_school"].astype(str).str.strip() == ""
    pff_work.loc[unresolved_name_mask, "mapped_school"] = pff_work.loc[unresolved_name_mask, "team_name"]

    pff_work["mapped_school_norm"] = pff_work["mapped_school"].map(normalize_school)

    pff_key_cols = ["Year", "mapped_school_norm", "side_norm", "name_norm"]
    roster_key_cols = ["Year", "school_norm", "side_norm", "name_norm"]

    pff_exact = pff_work[pff_key_cols + ["player", "player_id", "team_name", "mapped_school", "position", "player_game_count"]].copy()
    roster_exact = roster_work[roster_key_cols + ["Player", "Player_ID", "School", "pos"]].copy()

    bridge = pff_exact.merge(
        roster_exact,
        left_on=pff_key_cols,
        right_on=roster_key_cols,
        how="left",
        suffixes=("_pff", "_sr"),
    )

    bridge["match_layer"] = bridge["Player_ID"].map(
        lambda value: "exact" if pd.notna(value) and str(value).strip() != "" else "unmatched"
    )
    bridge["fuzzy_status"] = bridge["Player_ID"].map(
        lambda value: "exact" if pd.notna(value) and str(value).strip() != "" else "pending"
    )
    bridge["fuzzy_score"] = pd.NA

    roster_pool = roster_work[["Year", "school_norm", "side_norm", "name_norm", "Player", "Player_ID", "School", "pos"]].copy()
    roster_by_side = {
        key: group[["name_norm", "Player", "Player_ID", "School", "pos"]].to_dict("records")
        for key, group in roster_pool.groupby(["Year", "school_norm", "side_norm"], dropna=False)
    }

    unresolved_indices = bridge.index[bridge["match_layer"] == "unmatched"]
    for idx in unresolved_indices:
        year_val = bridge.at[idx, "Year"]
        school_norm_val = bridge.at[idx, "mapped_school_norm"]
        side_norm_val = bridge.at[idx, "side_norm"]
        name_norm_val = str(bridge.at[idx, "name_norm"])

        candidates = roster_by_side.get((year_val, school_norm_val, side_norm_val), [])
        if not candidates:
            bridge.at[idx, "fuzzy_status"] = "no_candidate"
            continue

        best_candidate = None
        best_score = -1
        for candidate in candidates:
            score = fuzz.token_set_ratio(name_norm_val, str(candidate.get("name_norm", "")))
            if score > best_score:
                best_score = score
                best_candidate = candidate

        if best_candidate is not None and best_score >= name_fuzzy_threshold:
            bridge.at[idx, "School"] = best_candidate.get("School", "")
            bridge.at[idx, "pos"] = best_candidate.get("pos", "")
            bridge.at[idx, "Player"] = best_candidate.get("Player", "")
            bridge.at[idx, "Player_ID"] = best_candidate.get("Player_ID", "")
            bridge.at[idx, "match_layer"] = "fuzzy_name"
            bridge.at[idx, "fuzzy_status"] = "matched"
            bridge.at[idx, "fuzzy_score"] = best_score
        else:
            bridge.at[idx, "fuzzy_status"] = "below_threshold"
            bridge.at[idx, "fuzzy_score"] = best_score if best_score >= 0 else pd.NA

    bridge = bridge[[
        "Year",
        "team_name",
        "mapped_school",
        "position",
        "player",
        "player_id",
        "player_game_count",
        "School",
        "pos",
        "Player",
        "Player_ID",
        "match_layer",
        "fuzzy_status",
        "fuzzy_score",
    ]]
    return bridge


tm = globals().get("team_map_skeleton")
if isinstance(tm, pd.DataFrame):
    team_map_for_match = tm.copy()
else:
    team_map_file = matching_data_dir / "pff_to_sportsref_team_map_skeleton.csv"
    if not team_map_file.exists():
        raise FileNotFoundError(f"Missing team map file: {team_map_file}")
    team_map_for_match = pd.read_csv(team_map_file, dtype=str).fillna("")

pff_matcher_skeleton_df = build_pff_matcher_skeleton(roster_master, pff_master, team_map_for_match)
pff_matcher_skeleton_path = matching_data_dir / "pff_to_sportsref_player_match_skeleton.csv"
pff_matcher_skeleton_df.to_csv(pff_matcher_skeleton_path, index=False, encoding="utf-8-sig")

print(f"PFF matcher skeleton rows: {len(pff_matcher_skeleton_df):,}")
print(f"Exact matched rows: {(pff_matcher_skeleton_df['match_layer'] == 'exact').sum():,}")
print(f"Fuzzy matched rows: {(pff_matcher_skeleton_df['match_layer'] == 'fuzzy_name').sum():,}")
print(f"Unmatched rows: {(pff_matcher_skeleton_df['match_layer'] == 'unmatched').sum():,}")
print(f"Saved: {pff_matcher_skeleton_path}")

PFF matcher skeleton rows: 110,944
Exact matched rows: 99,677
Fuzzy matched rows: 2,398
Unmatched rows: 8,869
Saved: g:\Other computers\Desktop\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\college\aggregated_stats\pff_to_sportsref_player_match_skeleton.csv


In [16]:
# -----------------------------
# 5) Build games-based player-year dataset (ready for external score notebook)
# -----------------------------

#Load pff matcher skeleton with match results for review and finalization
pff_matcher_skeleton_path = matching_data_dir / "pff_to_sportsref_player_match_skeleton.csv"
if not pff_matcher_skeleton_path.exists():
    raise FileNotFoundError(f"PFF matcher skeleton not found. Please run the team mapping and player matching sections first and save the skeleton to: {pff_matcher_skeleton_path}")
pff_matcher_skeleton_df = pd.read_csv(pff_matcher_skeleton_path, dtype=str).fillna("")

#Load roster_master from csv
roster_master_path = matching_data_dir / "roster_master.csv"
if not roster_master_path.exists():
    raise FileNotFoundError(f"Roster master file not found. Please run the roster loading section first and save the master roster to: {roster_master_path}")
roster_master = pd.read_csv(roster_master_path, dtype=str).fillna("")

def load_yearly_stat_file(directory: Path, suffix: str, value_columns: list[str], games_col: str = "games") -> pd.DataFrame:
    all_frames = []
    file_pattern = re.compile(rf"^(\d{{4}})_{suffix}\.csv$")

    for file_path in sorted(directory.glob(f"*_{suffix}.csv")):
        match = file_pattern.match(file_path.name)
        if not match:
            continue

        file_year = int(match.group(1))
        if file_year < YEAR_MIN or file_year > YEAR_MAX:
            continue

        df = pd.read_csv(file_path, dtype=str).fillna("")
        df["Year"] = pd.to_numeric(df.get("Year", file_year), errors="coerce").fillna(file_year).astype(int)

        keep_cols = ["Player_ID", "Player", "team_name_abbr", "conf_abbr", "Year"]
        if games_col in df.columns:
            keep_cols.append(games_col)
        if "awards" in df.columns:
            keep_cols.append("awards")
        keep_cols.extend([column for column in value_columns if column in df.columns])

        use_df = df[[column for column in keep_cols if column in df.columns]].copy()

        if games_col in use_df.columns:
            use_df.rename(columns={games_col: f"games_{suffix}"}, inplace=True)
        if "awards" in use_df.columns:
            use_df.rename(columns={"awards": f"awards_{suffix}"}, inplace=True)

        all_frames.append(use_df)

    if not all_frames:
        return pd.DataFrame()

    combined = pd.concat(all_frames, ignore_index=True)
    numeric_cols = [column for column in combined.columns if column.startswith("games_") or column in value_columns]
    for column in numeric_cols:
        if column in combined.columns:
            combined[column] = pd.to_numeric(combined[column], errors="coerce")
    return combined


passing_values = ["pass_cmp", "pass_att", "pass_yds", "pass_td", "pass_int", "pass_rating", "pass_yds_per_att"]
rushing_values = ["rush_att", "rush_yds", "rush_td", "yds_from_scrimmage", "scrim_td"]
receiving_values = ["rec", "rec_yds", "rec_td", "yds_from_scrimmage", "scrim_td"]
kicking_values = ["xpm", "xpa", "xp_pct", "fgm", "fga", "fg_pct", "kick_points"]
punting_values = ["punt", "punt_yds", "punt_yds_per_punt"]

passing_df = load_yearly_stat_file(passing_stats_dir, "passing", passing_values)
rushing_df = load_yearly_stat_file(rushing_stats_dir, "rushing", rushing_values)
receiving_df = load_yearly_stat_file(receiving_stats_dir, "receiving", receiving_values)
kicking_df = load_yearly_stat_file(kicking_stats_dir, "kicking", kicking_values)
punting_df = load_yearly_stat_file(punting_stats_dir, "punting", punting_values)

# Defense is school-year split files
defense_frames = []
defense_pattern = re.compile(r"^(.*)_(\d{4})_defense\.csv$")
for file_path in sorted(defensive_stats_dir.glob("*_defense.csv")):
    match = defense_pattern.match(file_path.name)
    if not match:
        continue

    file_year = int(match.group(2))
    if file_year < YEAR_MIN or file_year > YEAR_MAX:
        continue

    df = pd.read_csv(file_path, dtype=str).fillna("")
    keep_cols = [
        "Player_ID", "Player", "School", "Year", "games",
        "tackles_combined", "tackles_loss", "sacks", "def_int", "pass_defended", "fumbles_forced", "awards"
    ]
    use_df = df[[column for column in keep_cols if column in df.columns]].copy()
    use_df["Year"] = pd.to_numeric(use_df.get("Year", file_year), errors="coerce").fillna(file_year).astype(int)
    use_df.rename(columns={"games": "games_defense"}, inplace=True)
    if "awards" in use_df.columns:
        use_df.rename(columns={"awards": "awards_defense"}, inplace=True)

    for column in ["games_defense", "tackles_combined", "tackles_loss", "sacks", "def_int", "pass_defended", "fumbles_forced"]:
        if column in use_df.columns:
            use_df[column] = pd.to_numeric(use_df[column], errors="coerce")

    defense_frames.append(use_df)

defense_df = pd.concat(defense_frames, ignore_index=True) if defense_frames else pd.DataFrame()

awards_df = pd.read_csv(awards_stats_dir / "awards_player_year_consolidated.csv", dtype=str).fillna("")
awards_df = awards_df.rename(columns={"player_id": "Player_ID", "year": "Year", "award_count": "award_count"})
awards_df["Year"] = pd.to_numeric(awards_df["Year"], errors="coerce")
awards_df["award_count"] = pd.to_numeric(awards_df["award_count"], errors="coerce").fillna(0)
if "awards_won" not in awards_df.columns:
    awards_df["awards_won"] = ""
awards_df = awards_df[["Player_ID", "Year", "award_count", "awards_won"]]

standings_path = college_stats_dir / "standings" / "cfb_standings_2015_2025.csv"
if standings_path.exists():
    standings_df = pd.read_csv(standings_path, dtype=str).fillna("")
    standings_df["Year"] = pd.to_numeric(standings_df.get("year", pd.NA), errors="coerce")
    standings_df["sos"] = pd.to_numeric(standings_df.get("sos", pd.NA), errors="coerce")
    standings_df["school_norm"] = standings_df.get("school", "").map(normalize_school)
    standings_df = standings_df.dropna(subset=["Year"])
    standings_df["Year"] = standings_df["Year"].astype(int)
    standings_sos_df = standings_df[["Year", "school_norm", "sos"]].drop_duplicates(subset=["Year", "school_norm"], keep="first")
else:
    standings_sos_df = pd.DataFrame(columns=["Year", "school_norm", "sos"])
def split_award_tokens(value: str) -> list[str]:
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    parts = []
    for pipe_part in text.split("|"):
        for comma_part in pipe_part.split(","):
            token = comma_part.strip()
            token = re.sub(r"\s+", " ", token)
            token = token.strip("* ")
            if token:
                parts.append(token)
    return parts
def award_to_column_name(award_token: str) -> str:
    base = normalize_text(award_token).replace(" ", "_")
    base = re.sub(r"[^a-z0-9_]", "", base)
    base = re.sub(r"_+", "_", base).strip("_")
    if not base:
        base = "unknown"
    if base[0].isdigit():
        base = f"a_{base}"
    return f"award_{base}"
def extract_award_tokens(source_df: pd.DataFrame, awards_col: str) -> pd.DataFrame:
    if source_df is None or source_df.empty or awards_col not in source_df.columns:
        return pd.DataFrame(columns=["Player_ID", "Year", "award_token"])
    token_df = source_df[["Player_ID", "Year", awards_col]].copy()
    token_df["Player_ID"] = token_df["Player_ID"].astype(str).str.strip()
    token_df["Year"] = pd.to_numeric(token_df["Year"], errors="coerce")
    token_df = token_df.dropna(subset=["Year"])
    token_df["Year"] = token_df["Year"].astype(int)
    token_df = token_df[token_df["Player_ID"] != ""]
    token_df["award_token"] = token_df[awards_col].map(split_award_tokens)
    token_df = token_df.explode("award_token")
    token_df["award_token"] = token_df["award_token"].fillna("").astype(str).str.strip()
    token_df = token_df[token_df["award_token"] != ""]
    return token_df[["Player_ID", "Year", "award_token"]]
award_token_frames = [
    extract_award_tokens(passing_df, "awards_passing"),
    extract_award_tokens(rushing_df, "awards_rushing"),
    extract_award_tokens(receiving_df, "awards_receiving"),
    extract_award_tokens(defense_df, "awards_defense"),
    extract_award_tokens(kicking_df, "awards_kicking"),
    extract_award_tokens(punting_df, "awards_punting"),
    extract_award_tokens(awards_df, "awards_won"),
]
award_token_frames = [frame for frame in award_token_frames if not frame.empty]
if award_token_frames:
    awards_tokens_df = pd.concat(award_token_frames, ignore_index=True)
    awards_tokens_df["award_token"] = awards_tokens_df["award_token"].astype(str).str.strip()
    awards_tokens_df = awards_tokens_df[awards_tokens_df["award_token"] != ""]
    awards_tokens_df = awards_tokens_df.drop_duplicates(subset=["Player_ID", "Year", "award_token"])
else:
    awards_tokens_df = pd.DataFrame(columns=["Player_ID", "Year", "award_token"])
awards_player_year_df = pd.DataFrame(columns=["Player_ID", "Year", "awards", "award_count", "heisman_finish"])
award_binary_columns = []
if not awards_tokens_df.empty:
    awards_tokens_df["heisman_finish"] = awards_tokens_df["award_token"].map(lambda token: int(re.match(r"^H-(\d+)$", token, flags=re.IGNORECASE).group(1)) if re.match(r"^H-(\d+)$", token, flags=re.IGNORECASE) else pd.NA)

    awards_raw_df = awards_tokens_df.groupby(["Player_ID", "Year"], as_index=False).agg({
        "award_token": lambda values: "|".join(sorted(set(values))),
    }).rename(columns={"award_token": "awards"})

    award_count_df = awards_tokens_df.groupby(["Player_ID", "Year"], as_index=False).agg({
        "award_token": "nunique",
    }).rename(columns={"award_token": "award_count"})

    heisman_df = awards_tokens_df.dropna(subset=["heisman_finish"]).groupby(["Player_ID", "Year"], as_index=False).agg({
        "heisman_finish": "min",
    })

    non_heisman_df = awards_tokens_df[~awards_tokens_df["award_token"].str.match(r"^H-\d+$", case=False, na=False)].copy()
    if not non_heisman_df.empty:
        non_heisman_df["value"] = 1
        binary_matrix = non_heisman_df.pivot_table(index=["Player_ID", "Year"], columns="award_token", values="value", aggfunc="max", fill_value=0).reset_index()
        token_columns = [column for column in binary_matrix.columns if column not in ["Player_ID", "Year"]]
        rename_map = {}
        used_cols = set()
        for token in token_columns:
            base_col = award_to_column_name(token)
            col_name = base_col
            suffix = 2
            while col_name in used_cols:
                col_name = f"{base_col}_{suffix}"
                suffix += 1
            rename_map[token] = col_name
            used_cols.add(col_name)
        binary_matrix = binary_matrix.rename(columns=rename_map)
        award_binary_columns = [column for column in binary_matrix.columns if column not in ["Player_ID", "Year"]]
    else:
        binary_matrix = awards_raw_df[["Player_ID", "Year"]].copy()

    awards_player_year_df = awards_raw_df.merge(award_count_df, on=["Player_ID", "Year"], how="left")
    awards_player_year_df = awards_player_year_df.merge(heisman_df, on=["Player_ID", "Year"], how="left")
    awards_player_year_df = awards_player_year_df.merge(binary_matrix, on=["Player_ID", "Year"], how="left")
    awards_player_year_df["award_count"] = pd.to_numeric(awards_player_year_df["award_count"], errors="coerce").fillna(0).astype(int)
    awards_player_year_df["heisman_finish"] = pd.to_numeric(awards_player_year_df["heisman_finish"], errors="coerce").fillna(0).astype(int)
    for award_col in award_binary_columns:
        awards_player_year_df[award_col] = pd.to_numeric(awards_player_year_df[award_col], errors="coerce").fillna(0).astype(int)

# PFF games through skeleton bridge
pff_games_df = pff_matcher_skeleton_df[["Player_ID", "Year", "player_game_count"]].copy()
pff_games_df["Year"] = pd.to_numeric(pff_games_df["Year"], errors="coerce")
pff_games_df["player_game_count"] = pd.to_numeric(pff_games_df["player_game_count"], errors="coerce")
pff_games_df = pff_games_df.rename(columns={"player_game_count": "games_pff"})
pff_games_df = pff_games_df[
    pff_games_df["Player_ID"].notna() & (pff_games_df["Player_ID"].astype(str).str.strip() != "")
]

# Base roster key (player-year-position)
base_cols = ["Player_ID", "Player", "School", "Year", "pos", "pos_norm"]
player_year_base = roster_master[base_cols].copy()
player_year_base["Player_ID"] = player_year_base["Player_ID"].astype(str).str.strip()
player_year_base["Year"] = pd.to_numeric(player_year_base["Year"], errors="coerce")
player_year_base = player_year_base.dropna(subset=["Year"])
player_year_base["Year"] = player_year_base["Year"].astype(int)
player_year_base = player_year_base[(player_year_base["Year"] >= YEAR_MIN) & (player_year_base["Year"] <= YEAR_MAX)]
player_year_base["school_norm"] = player_year_base["School"].map(normalize_school)
if not standings_sos_df.empty:
    player_year_base = player_year_base.merge(standings_sos_df, on=["Year", "school_norm"], how="left")
else:
    player_year_base["sos"] = pd.NA
player_year_base = player_year_base.drop(columns=["school_norm"], errors="ignore")

# Merge source blocks by Player_ID + Year without identity-column collisions
merge_keys = ["Player_ID", "Year"]
identity_drop_cols = {
    "Player", "School", "pos", "team_name_abbr", "conf_abbr",
    "name_norm", "school_norm", "pos_norm", "source_file",
    "player", "player_id", "position", "mapped_school", "team_name",
    "awards_passing", "awards_rushing", "awards_receiving", "awards_defense", "awards_kicking", "awards_punting", "awards_won"
}

for source_df in [passing_df, rushing_df, receiving_df, defense_df, kicking_df, punting_df, pff_games_df]:
    if source_df is None or source_df.empty:
        continue

    available_keys = [key for key in merge_keys if key in source_df.columns]
    if len(available_keys) != 2:
        continue

    source_use = source_df.copy()
    source_non_keys = [column for column in source_use.columns if column not in merge_keys]
    source_non_keys = [column for column in source_non_keys if column not in identity_drop_cols]
    source_non_keys = [column for column in source_non_keys if column not in player_year_base.columns]

    if not source_non_keys:
        continue

    source_use = source_use[merge_keys + source_non_keys]
    source_use["Player_ID"] = source_use["Player_ID"].astype(str).str.strip()
    source_use["Year"] = pd.to_numeric(source_use["Year"], errors="coerce")
    source_use = source_use.dropna(subset=["Year"])
    source_use["Year"] = source_use["Year"].astype(int)
    dedup_df = source_use.drop_duplicates(subset=merge_keys, keep="first")
    player_year_base = player_year_base.merge(dedup_df, on=merge_keys, how="left")
if not awards_player_year_df.empty:
    player_year_base = player_year_base.merge(awards_player_year_df, on=merge_keys, how="left")

# Single games metric (games only, no starts)
game_columns = [
    column for column in ["games_passing", "games_rushing", "games_receiving", "games_kicking", "games_punting", "games_defense", "games_pff"]
    if column in player_year_base.columns
]
if game_columns:
    player_year_base["games_total"] = player_year_base[game_columns].max(axis=1, skipna=True)
else:
    player_year_base["games_total"] = pd.NA

player_year_base["games_total"] = pd.to_numeric(player_year_base["games_total"], errors="coerce").fillna(0)
player_year_base["award_count"] = pd.to_numeric(player_year_base.get("award_count", 0), errors="coerce").fillna(0)
player_year_base["sos"] = pd.to_numeric(player_year_base.get("sos", pd.NA), errors="coerce")
player_year_base["heisman_finish"] = pd.to_numeric(player_year_base.get("heisman_finish", 0), errors="coerce").fillna(0).astype(int)
player_year_base["awards"] = player_year_base.get("awards", "").fillna("").astype(str)
for award_col in award_binary_columns:
    if award_col in player_year_base.columns:
        player_year_base[award_col] = pd.to_numeric(player_year_base[award_col], errors="coerce").fillna(0).astype(int)

# Keep only normalized final position groups from POS_MAP
player_year_ready = player_year_base[player_year_base["pos_norm"].isin(FINAL_POSITION_GROUPS)].copy()

identity_columns = ["Player_ID", "Player", "School", "Year", "pos", "pos_norm"]
shared_columns = ["games_total", "award_count", "sos"]
award_export_columns = ["awards", "heisman_finish"] + award_binary_columns
position_stat_columns = {
    "QB": ["games_passing", "games_rushing", "pass_cmp", "pass_att", "pass_yds", "pass_td", "pass_int", "pass_rating", "pass_yds_per_att", "rush_att", "rush_yds", "rush_td", "yds_from_scrimmage", "scrim_td"],
    "RB": ["games_rushing", "games_receiving", "rush_att", "rush_yds", "rush_td", "rec", "rec_yds", "rec_td", "yds_from_scrimmage", "scrim_td"],
    "WR": ["games_rushing", "games_receiving", "rush_att", "rush_yds", "rush_td", "rec", "rec_yds", "rec_td", "yds_from_scrimmage", "scrim_td"],
    "TE": ["games_rushing", "games_receiving", "rush_att", "rush_yds", "rush_td", "rec", "rec_yds", "rec_td", "yds_from_scrimmage", "scrim_td"],
    "DB": ["games_defense", "games_pff", "tackles_combined", "tackles_loss", "sacks", "def_int", "pass_defended", "fumbles_forced"],
    "EDGE": ["games_defense", "games_pff", "tackles_combined", "tackles_loss", "sacks", "def_int", "pass_defended", "fumbles_forced"],
    "IDL": ["games_defense", "games_pff", "tackles_combined", "tackles_loss", "sacks", "def_int", "pass_defended", "fumbles_forced"],
    "LB": ["games_defense", "games_pff", "tackles_combined", "tackles_loss", "sacks", "def_int", "pass_defended", "fumbles_forced"],
    "OL": ["games_pff"],
    "SPEC": ["games_kicking", "games_punting", "xpm", "xpa", "xp_pct", "fgm", "fga", "fg_pct", "kick_points", "punt", "punt_yds", "punt_yds_per_punt"],
}

# Save master player-year output
player_year_master_path = modeling_data_dir / "college_player_year_master_ready.csv"
master_year_exclude_cols = set([column for column in player_year_ready.columns if column.startswith("award_")]) | {"awards", "heisman_finish"}
master_year_columns = [column for column in player_year_ready.columns if column not in master_year_exclude_cols]
player_year_ready[master_year_columns].to_csv(player_year_master_path, index=False, encoding="utf-8-sig")

# Save per-position player-year outputs
position_counts = {}
for position in FINAL_POSITION_GROUPS:
    position_df = player_year_ready[player_year_ready["pos_norm"] == position].copy()
    if position_df.empty:
        continue
    selected_columns = identity_columns + shared_columns + position_stat_columns.get(position, []) + award_export_columns
    selected_columns = [column for column in selected_columns if column in position_df.columns]
    output_path = modeling_data_dir / f"{position}_college_player_year_ready.csv"
    position_df[selected_columns].sort_values(["Year", "School", "Player"]).to_csv(output_path, index=False, encoding="utf-8-sig")
    position_counts[position] = len(position_df)

# Also save a compact career-level aggregation by position (sum of production + games + awards)
production_columns = [
    "pass_cmp", "pass_att", "pass_yds", "pass_td", "pass_int",
    "rush_att", "rush_yds", "rush_td", "rec", "rec_yds", "rec_td",
    "tackles_combined", "tackles_loss", "sacks", "def_int", "pass_defended", "fumbles_forced",
    "yds_from_scrimmage", "scrim_td",
    "xpm", "xpa", "xp_pct", "fgm", "fga", "fg_pct", "kick_points",
    "punt", "punt_yds", "punt_yds_per_punt",
    "games_passing", "games_rushing", "games_receiving", "games_kicking", "games_punting", "games_defense", "games_pff"
]
production_columns = [column for column in production_columns if column in player_year_ready.columns]
def merge_awards_for_career(series: pd.Series) -> str:
    all_tokens = set()
    for value in series.dropna().astype(str):
        for token in split_award_tokens(value):
            all_tokens.add(token)
    return "|".join(sorted(all_tokens))
def best_heisman_finish(series: pd.Series) -> int:
    values = pd.to_numeric(series, errors="coerce")
    values = values[values > 0]
    return int(values.min()) if len(values) else 0
def merge_schools_played(series: pd.Series) -> str:
    ordered_unique = []
    seen = set()
    for value in series.dropna().astype(str):
        school = value.strip()
        if not school or school in seen:
            continue
        seen.add(school)
        ordered_unique.append(school)
    return "|".join(ordered_unique)

career_group_cols = ["Player_ID", "Player", "pos_norm"]
career_work_df = player_year_ready.copy()
career_work_df["sos"] = pd.to_numeric(career_work_df.get("sos", pd.NA), errors="coerce").fillna(0)
career_work_df["games_total"] = pd.to_numeric(career_work_df.get("games_total", 0), errors="coerce").fillna(0)
career_work_df["sos_games_weight"] = career_work_df["sos"] * career_work_df["games_total"]
career_agg = {
    "games_total": "sum",
    "award_count": "sum",
    "Year": "count",
    "awards": merge_awards_for_career,
    "heisman_finish": best_heisman_finish,
}
for column in production_columns:
    career_agg[column] = "sum"
for award_col in award_binary_columns:
    if award_col in player_year_ready.columns:
        career_agg[award_col] = "max"

career_ready = career_work_df.groupby(career_group_cols, as_index=False).agg(career_agg)
career_ready = career_ready.rename(columns={"Year": "seasons_count"})
schools_played_df = career_work_df.groupby(career_group_cols, as_index=False).agg({"School": merge_schools_played}).rename(columns={"School": "schools_played"})
latest_school_df = career_work_df.sort_values(["Year"]).groupby(career_group_cols, as_index=False).tail(1)[career_group_cols + ["School"]].rename(columns={"School": "latest_school"})
sos_career_df = career_work_df.groupby(career_group_cols, as_index=False).agg({"sos_games_weight": "sum", "games_total": "sum"})
sos_career_df["sos_season_weighted"] = sos_career_df["sos_games_weight"] / sos_career_df["games_total"].replace(0, pd.NA)
sos_career_df["sos_season_weighted"] = pd.to_numeric(sos_career_df["sos_season_weighted"], errors="coerce")
sos_career_df = sos_career_df[career_group_cols + ["sos_season_weighted"]]
career_ready = career_ready.merge(schools_played_df, on=career_group_cols, how="left")
career_ready = career_ready.merge(latest_school_df, on=career_group_cols, how="left")
career_ready = career_ready.merge(sos_career_df, on=career_group_cols, how="left")

career_master_path = modeling_data_dir / "college_player_career_master_ready.csv"
master_career_exclude_cols = set([column for column in career_ready.columns if column.startswith("award_")]) | {"awards", "heisman_finish"}
master_career_columns = [column for column in career_ready.columns if column not in master_career_exclude_cols]
career_ready[master_career_columns].to_csv(career_master_path, index=False, encoding="utf-8-sig")

for position in FINAL_POSITION_GROUPS:
    position_df = career_ready[career_ready["pos_norm"] == position].copy()
    if position_df.empty:
        continue
    position_columns = ["Player_ID", "Player", "pos_norm", "latest_school", "schools_played", "seasons_count", "games_total", "sos_season_weighted", "award_count", "awards", "heisman_finish"] + position_stat_columns.get(position, []) + award_binary_columns
    position_columns = [column for column in position_columns if column in position_df.columns]
    output_path = modeling_data_dir / f"{position}_college_player_career_ready.csv"
    position_df[position_columns].sort_values(["games_total", "award_count"], ascending=[False, False]).to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Player-year rows (ready): {len(player_year_ready):,}")
print(f"Career rows (ready): {len(career_ready):,}")
print(f"Saved: {player_year_master_path}")
print(f"Saved: {career_master_path}")
print("Per-position player-year row counts:")
print(position_counts)

Player-year rows (ready): 161,041
Career rows (ready): 68,467
Saved: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\college\aggregated_stats\college_player_year_master_ready.csv
Saved: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\college\aggregated_stats\college_player_career_master_ready.csv
Per-position player-year row counts:
{'DB': 29213, 'EDGE': 5447, 'IDL': 18435, 'LB': 20598, 'OL': 25237, 'QB': 7883, 'RB': 12128, 'SPEC': 10809, 'TE': 9297, 'WR': 21994}


In [17]:
# -----------------------------
# 6) Drop all-zero award binary columns from per-position exports
# -----------------------------

def drop_all_zero_award_binary_columns(file_path: Path) -> tuple[int, list[str]]:
    if not file_path.exists():
        return 0, []

    df = pd.read_csv(file_path, dtype=str).fillna("")
    award_binary_cols = [
        column
        for column in df.columns
        if column.startswith("award_") and column != "award_count"
    ]
    if not award_binary_cols:
        return 0, []

    removed_cols = []
    for column in award_binary_cols:
        col_values = pd.to_numeric(df[column], errors="coerce").fillna(0)
        if (col_values == 0).all():
            removed_cols.append(column)

    if removed_cols:
        df = df.drop(columns=removed_cols)
        df.to_csv(file_path, index=False, encoding="utf-8-sig")

    return len(removed_cols), removed_cols

cleanup_summary = []
for position in FINAL_POSITION_GROUPS:
    for suffix in ["player_year", "player_career"]:
        file_path = modeling_data_dir / f"{position}_college_{suffix}_ready.csv"
        removed_count, removed_cols = drop_all_zero_award_binary_columns(file_path)
        cleanup_summary.append({
            "file": file_path.name,
            "removed_count": removed_count,
            "removed_cols": removed_cols,
        })

for item in cleanup_summary:
    print(f"{item['file']}: removed {item['removed_count']} all-zero award columns")
    if item["removed_cols"]:
        print(f"  - {', '.join(item['removed_cols'])}")

DB_college_player_year_ready.csv: removed 37 all-zero award columns
  - award_aac_offensive_player_of_the_year, award_acc_offensive_player_of_the_year, award_acc_player_of_the_year, award_big_12_offensive_player_of_the_year, award_big_east_defensive_player_of_the_year, award_big_east_offensive_player_of_the_year, award_big_ten_offensive_player_of_the_year, award_big_ten_player_of_the_year, award_cusa_most_valuable_player, award_cusa_offensive_player_of_the_year, award_dave_rimington_trophy_most_outstanding_center, award_davey_o_brien_award_most_outstanding_quarterback, award_dick_butkus_award_most_outstanding_linebacker, award_doak_walker_award_most_outstanding_running_back, award_john_mackey_award_most_outstanding_tight_end, award_john_outland_trophy_most_outstanding_interior_lineman, award_johnny_unitas_golden_arm_award_most_outstanding_senior_quarterback, award_lou_groza_award_most_outstanding_place_kicker, award_mac_defensive_player_of_the_year, award_mac_offensive_player_of_the_ye